# Validación StationLINE

Comprueba paso a paso que el módulo LINE funciona correctamente:
1. Carga de grafos (metro y cercanías)
2. Inspección del grafo y distribución de pesos
3. Matriz de adyacencia simétrica (similitud temporal)
4. Entrenamiento LINE — first-order + second-order
5. Validación de embeddings
6. Visualización PCA (coloreada por línea)
7. Expansión a punto_acceso
8. Guardar embeddings

In [ ]:
import sys
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from sklearn.decomposition import PCA

sys.path.insert(0, '.')
from station_line import StationLINE

CHECK_DIR = Path('../../cubo_izquierdo/check_transbordo')

with open(CHECK_DIR / 'metro_multidigraph.pkl', 'rb') as f:
    G_metro = pickle.load(f)

with open(CHECK_DIR / 'cercanias_multidigraph_sin_civis.pkl', 'rb') as f:
    G_cercanias = pickle.load(f)

print('Módulo importado correctamente')
print(f'Metro:     {G_metro.number_of_nodes()} nodos, {G_metro.number_of_edges()} aristas')
print(f'Cercanías: {G_cercanias.number_of_nodes()} nodos, {G_cercanias.number_of_edges()} aristas')

---
## 1 — Inspección del grafo

In [ ]:
# Atributos de nodo y arista
for nombre, G in [('Metro', G_metro), ('Cercanías', G_cercanias)]:
    nodo = list(G.nodes(data=True))[0]
    arista = list(G.edges(data=True))[0]
    print(f'=== {nombre} ===')
    print(f'  Ejemplo nodo:   {nodo[0]}')
    print(f'  Atributos:      {nodo[1]}')
    print(f'  Ejemplo arista: {arista[0]} → {arista[1]}')
    print(f'  Atributos:      {arista[2]}')
    print()

In [ ]:
# Distribución de pesos (tiempos de viaje)
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

for ax, (nombre, G) in zip(axes, [('Metro', G_metro), ('Cercanías', G_cercanias)]):
    pesos = [d.get('weight', 0) for _, _, d in G.edges(data=True)]
    ax.hist(pesos, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
    ax.set_xlabel('Tiempo de viaje (min)')
    ax.set_ylabel('Nº aristas')
    ax.set_title(f'{nombre} — distribución de tiempos\n'
                 f'(media={np.mean(pesos):.1f} min, máx={np.max(pesos):.1f} min)')

plt.tight_layout()
plt.show()

---
## 2 — Matriz de adyacencia

In [ ]:
# Inspeccionar la matriz de adyacencia que construye el modelo
# (simetrica, ponderada por 1/tiempo)
model_tmp = StationLINE(dim=4, num_epochs_fo=1, num_epochs_so=1)

for nombre, G in [('Metro', G_metro), ('Cercanías', G_cercanias)]:
    nodes, adj = model_tmp._build_adjacency(G)
    n = len(nodes)
    n_conexiones = int((np.triu(adj, k=1) > 0).sum())
    pesos_sim = adj[adj > 0]

    print(f'=== {nombre} ===')
    print(f'  Shape:            {adj.shape}')
    print(f'  Simétrica:        {np.allclose(adj, adj.T)}')
    print(f'  Pares conectados: {n_conexiones} / {n*(n-1)//2} posibles ({n_conexiones/(n*(n-1)//2)*100:.1f}%)')
    print(f'  Similitud mín:    {pesos_sim.min():.4f}  (= 1/tiempo_máx)')
    print(f'  Similitud máx:    {pesos_sim.max():.4f}  (= 1/tiempo_mín)')
    print(f'  Similitud media:  {pesos_sim.mean():.4f}')
    print()

In [ ]:
PROBANDO LOS MEJORES STATE_INIT PARA METRO Y CERCANIAS

In [ ]:
# Barrido init_scale — Metro
# Comprueba cómo afecta la escala de inicialización a la varianza FO y a la
# correlación Spearman tiempo↔distancia. Ejecutar una vez para elegir el valor.

from scipy.stats import spearmanr
from scipy.spatial.distance import cdist

print('BARRIDO init_scale (Metro, num_negative=1)\n')
print(f'{"init_scale":>12}  {"var_FO":>10}  {"var_SO":>10}  {"r_Spearman":>12}  {"p":>10}')
print('-' * 62)

for scale in [0.01, 0.05, 0.1, 0.4,0.5,0.6,0.7, 1.0]:
    m = StationLINE(dim=32, num_negative=1, lr=0.075,
                    num_epochs_fo=900, num_epochs_so=150,
                    init_scale=scale, random_state=42)
    emb = m.fit_transform(G_metro)

    fo_cols = [c for c in emb.columns if c.startswith('fo_')]
    so_cols = [c for c in emb.columns if c.startswith('so_')]
    var_fo = emb[fo_cols].values.var()
    var_so = emb[so_cols].values.var()

    with np.errstate(divide='ignore', invalid='ignore'):
        T = np.where(m.adj_ > 0, 1.0 / m.adj_, 0.0)
    X = emb.values.astype(float)
    D = cdist(X, X, metric='cosine')
    idx = np.triu_indices(len(m.station_index_), k=1)
    times, dists = T[idx], D[idx]
    mask = times > 0
    r, p = spearmanr(times[mask], dists[mask])

    print(f'{scale:>12.2f}  {var_fo:>10.5f}  {var_so:>10.5f}  {r:>12.4f}  {p:>10.2e}')

## CERCANIAS PROBANOD STATE INIT ##

In [ ]:
# Barrido init_scale — CERCANIAS
# Comprueba cómo afecta la escala de inicialización a la varianza FO y a la
# correlación Spearman tiempo↔distancia. Ejecutar una vez para elegir el valor.

from scipy.stats import spearmanr
from scipy.spatial.distance import cdist

print('BARRIDO init_scale (CERCANIAS\n')
print(f'{"init_scale":>12}  {"var_FO":>10}  {"var_SO":>10}  {"r_Spearman":>12}  {"p":>10}')
print('-' * 62)

for scale in [0.01, 0.05, 0.1,0.25,0.4,0.5,0.6, 1.0]:
    m = StationLINE(dim=32, num_negative=1, lr=0.05,
                    num_epochs_fo=400, num_epochs_so=400,
                    init_scale=scale, random_state=42)
    emb = m.fit_transform(G_cercanias)

    fo_cols = [c for c in emb.columns if c.startswith('fo_')]
    so_cols = [c for c in emb.columns if c.startswith('so_')]
    var_fo = emb[fo_cols].values.var()
    var_so = emb[so_cols].values.var()

    with np.errstate(divide='ignore', invalid='ignore'):
        T = np.where(m.adj_ > 0, 1.0 / m.adj_, 0.0
                    )
    X = emb.values.astype(float)
    D = cdist(X, X, metric='cosine')
    idx = np.triu_indices(len(m.station_index_), k=1)
    times, dists = T[idx], D[idx]
    mask = times > 0
    r, p = spearmanr(times[mask], dists[mask])

    print(f'{scale:>12.2f}  {var_fo:>10.5f}  {var_so:>10.5f}  {r:>12.4f}  {p:>10.2e}')

In [ ]:
import time

# num_epochs_fo / num_epochs_so / init_scale elegidos tras barrido:
#
# Metro (270 aristas, num_negative=1):
#   El update es  emb[i] += g * u_j  →  proporcional a |u_j|.
#   Con init_scale=0.01 los embeddings parten tan cerca de 0 que los pasos
#   son microscópicos y FO no diverge del origen (varianza FO≈0.001).
#   Con init_scale=0.1 los pasos son 10× mayores y FO converge bien.
#
# Cercanías (96 aristas, num_negative=1):
#   Grafo extremadamente sparse (casi árbol) → SO sin señal útil.
#   FO domina; init_scale=0.1 también mejora su dispersión.

model_metro = StationLINE(
    dim=32,
    num_negative=1,
    lr=0.075,
    num_epochs_fo=900,
    num_epochs_so=150,
    init_scale=0.6,
    random_state=42,
)
t0 = time.time()
emb_metro = model_metro.fit_transform(G_metro)
print(f'Metro      → {emb_metro.shape}  ({time.time()-t0:.1f}s)\n')

model_cercanias = StationLINE(
    dim=32,
    num_negative=1,
    lr=0.05,
    num_epochs_fo=500,
    num_epochs_so=500,
    init_scale=0.5,
    random_state=42,
)
t0 = time.time()
emb_cercanias = model_cercanias.fit_transform(G_cercanias)
print(f'Cercanías  → {emb_cercanias.shape}  ({time.time()-t0:.1f}s)')

---
## 4 — Validación de embeddings

In [ ]:
# CHECK 1: shape correcta
for nombre, G, emb in [('Metro', G_metro, emb_metro), ('Cercanías', G_cercanias, emb_cercanias)]:
    n_nodos = G.number_of_nodes()
    ok_filas = len(emb) == n_nodos
    ok_cols  = emb.shape[1] == 2 * model_metro.dim
    print(f'{nombre}: shape {emb.shape} → filas={"✓" if ok_filas else "✗"} cols={"✓" if ok_cols else "✗"}')

print()

# CHECK 2: sin NaN ni Inf
for nombre, emb in [('Metro', emb_metro), ('Cercanías', emb_cercanias)]:
    ok = not emb.isnull().any().any() and np.isfinite(emb.values).all()
    print(f'{nombre}: sin NaN/Inf → {"✓" if ok else "✗"}')

print()

# CHECK 3: first-order y second-order tienen varianza (no colapsan a 0)
for nombre, emb in [('Metro', emb_metro), ('Cercanías', emb_cercanias)]:
    fo_cols = [c for c in emb.columns if c.startswith('fo_')]
    so_cols = [c for c in emb.columns if c.startswith('so_')]
    var_fo = emb[fo_cols].values.var()
    var_so = emb[so_cols].values.var()
    print(f'{nombre}: varianza FO={var_fo:.5f}  SO={var_so:.5f}')

In [ ]:
# CHECK 4: estaciones de la misma línea deben tener embeddings similares (first-order)
# Verificar que la similitud coseno media intra-línea > inter-línea
from sklearn.metrics.pairwise import cosine_similarity

fo_cols = [c for c in emb_metro.columns if c.startswith('fo_')]
X_fo = emb_metro[fo_cols].values

# Extraer línea de cada estación a partir de sus access_point_codes
def get_line(node, G):
    aps = G.nodes[node].get('access_point_codes', set())
    if isinstance(aps, str): aps = {aps}
    # Tomar línea del primer AP: 02_L3_P10 → L3
    for ap in aps:
        parts = ap.split('_')
        if len(parts) >= 2: return parts[1]
    return 'unknown'

lines = [get_line(s, G_metro) for s in emb_metro.index]
emb_metro['_line'] = lines

print('Similitud coseno media intra-línea vs inter-línea (first-order):')
print('(esperado: intra > inter — misma línea = vectores más similares)\n')

sim_matrix = cosine_similarity(X_fo)
lines_arr = np.array(lines)

intra_sims, inter_sims = [], []
n = len(lines_arr)
for i in range(n):
    for j in range(i+1, n):
        if lines_arr[i] == lines_arr[j]:
            intra_sims.append(sim_matrix[i, j])
        else:
            inter_sims.append(sim_matrix[i, j])

intra_mean = np.mean(intra_sims)
inter_mean = np.mean(inter_sims)
print(f'  Similitud media intra-línea: {intra_mean:.4f}')
print(f'  Similitud media inter-línea: {inter_mean:.4f}')
print(f'  intra > inter → {"✓" if intra_mean > inter_mean else "✗"}')

# Eliminar _line del DataFrame Y del estado interno del modelo
emb_metro = emb_metro.drop(columns=['_line'])
model_metro.embeddings_ = emb_metro

---
## 5 — Visualización PCA

In [ ]:
# CHECK 5: correlación distancia topológica ↔ distancia en embedding
# Si LINE funciona bien, estaciones topológicamente cercanas (poco tiempo de viaje)
# deben tener embeddings más similares → correlación negativa tiempo/distancia_coseno

from scipy.stats import spearmanr
from scipy.spatial.distance import cdist

print('Correlación de Spearman: tiempo de viaje vs distancia coseno en embedding\n')

for nombre, model, emb in [('Metro', model_metro, emb_metro), ('Cercanías', model_cercanias, emb_cercanias)]:
    n = len(model.station_index_)

    # Tiempo de viaje (adj guarda similitud = 1/tiempo → tiempo = 1/sim, 0 si no existe)
    with np.errstate(divide='ignore', invalid='ignore'):
        T = np.where(model.adj_ > 0, 1.0 / model.adj_, 0.0)

    # Distancia coseno en embedding (fo + so concatenados)
    X = emb[[c for c in emb.columns if c.startswith('fo_') or c.startswith('so_')]].values.astype(float)
    D = cdist(X, X, metric='cosine')

    # Triángulo superior (sin diagonal)
    idx = np.triu_indices(n, k=1)
    times = T[idx]
    dists = D[idx]

    # Solo pares conectados (tiempo > 0)
    mask = times > 0
    r, p = spearmanr(times[mask], dists[mask])

    print(f'  {nombre}: r={r:.3f}  (p={p:.2e})  sobre {mask.sum():,} pares')
    if r > 0:
        print(f'    ✓ correlación positiva: más tiempo → más distancia en embedding')
    else:
        print(f'    ✗ correlación negativa — LINE no preserva distancias topológicas')
print()

In [ ]:
# CHECK 6: distribución intra vs inter-línea (histograma)
# Muestra si las distribuciones se separan bien o se solapan demasiado

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

fig_hist = make_subplots(rows=1, cols=2, subplot_titles=['Metro', 'Cercanías'])

for col_idx, (nombre, G, emb) in enumerate([('Metro', G_metro, emb_metro), ('Cercanías', G_cercanias, emb_cercanias)], start=1):
    cols = [c for c in emb.columns if c.startswith('fo_') or c.startswith('so_')]
    X = emb[cols].values.astype(float)
    sim = cos_sim(X)
    lines_arr = np.array([get_line(s, G) for s in emb.index])
    n = len(lines_arr)

    intra, inter = [], []
    for i in range(n):
        for j in range(i+1, n):
            (intra if lines_arr[i] == lines_arr[j] else inter).append(sim[i, j])

    fig_hist.add_trace(go.Histogram(x=intra, name='intra-línea', opacity=0.6,
                                    marker_color='steelblue', nbinsx=50,
                                    legendgroup='intra', showlegend=(col_idx==1)), row=1, col=col_idx)
    fig_hist.add_trace(go.Histogram(x=inter, name='inter-línea', opacity=0.6,
                                    marker_color='tomato', nbinsx=50,
                                    legendgroup='inter', showlegend=(col_idx==1)), row=1, col=col_idx)

fig_hist.update_layout(barmode='overlay', height=400,
                       title='Distribución similitud coseno intra vs inter-línea',
                       hovermode='x')
HTML(fig_hist.to_html(include_plotlyjs='cdn', full_html=False))

In [ ]:
# CHECK 7: vecinos más cercanos para estaciones conocidas
# Valida cualitativamente que el embedding tiene sentido topológico

from sklearn.metrics.pairwise import cosine_similarity as cos_sim

HUBS = ['SOL', 'ATOCHA', 'NUEVOS MINISTERIOS', 'CHAMARTIN', 'OPERA', 'PRINCIPE PIO']
K = 8

print(f'Top-{K} vecinos más cercanos en embedding (metro)\n')

cols = [c for c in emb_metro.columns if c.startswith('fo_') or c.startswith('so_')]
X = emb_metro[cols].values.astype(float)
sim_matrix = cos_sim(X)
station_list = list(emb_metro.index)

for hub in HUBS:
    # Buscar la estación (puede ser match parcial)
    matches = [s for s in station_list if hub in s.upper()]
    if not matches:
        continue
    station = matches[0]
    idx = station_list.index(station)
    sims = sim_matrix[idx]
    top_idx = np.argsort(sims)[::-1][1:K+1]  # excluir la propia estación
    print(f'  {station}:')
    for rank, j in enumerate(top_idx, 1):
        print(f'    {rank}. {station_list[j]:<40} sim={sims[j]:.4f}')
    print()

In [ ]:
# CHECK 7.2: vecinos más cercanos para estaciones conocidas
# Valida cualitativamente que el embedding tiene sentido topológico

from sklearn.metrics.pairwise import cosine_similarity as cos_sim

HUBS = ['SOL', 'ATOCHA', 'NUEVOS MINISTERIOS', 'CHAMARTIN', 'PRINCIPE PIO']
K = 8

print(f'Top-{K} vecinos más cercanos en embedding (cercanias)\n')

cols = [c for c in emb_cercanias.columns if c.startswith('fo_') or c.startswith('so_')]
X = emb_cercanias[cols].values.astype(float)
sim_matrix = cos_sim(X)
station_list = list(emb_cercanias.index)

for hub in HUBS:
    # Buscar la estación (puede ser match parcial)
    matches = [s for s in station_list if hub in s.upper()]
    if not matches:
        continue
    station = matches[0]
    idx = station_list.index(station)
    sims = sim_matrix[idx]
    top_idx = np.argsort(sims)[::-1][1:K+1]  # excluir la propia estación
    print(f'  {station}:')
    for rank, j in enumerate(top_idx, 1):
        print(f'    {rank}. {station_list[j]:<40} sim={sims[j]:.4f}')
    print()

In [ ]:
# CHECK 8: heatmap de similitudes ordenado por línea
# Si LINE funciona bien debe verse estructura de bloques (diagonal por línea)

from sklearn.metrics.pairwise import cosine_similarity as cos_sim

for nombre, G, emb in [('Metro', G_metro, emb_metro), ('Cercanías', G_cercanias, emb_cercanias)]:
    cols = [c for c in emb.columns if c.startswith('fo_') or c.startswith('so_')]
    X = emb[cols].values.astype(float)

    lines_list = [get_line(s, G) for s in emb.index]
    stations = list(emb.index)

    # Ordenar por línea para ver bloques
    order = np.argsort(lines_list)
    X_sorted = X[order]
    labels_sorted = [f'{stations[i]} ({lines_list[i]})' for i in order]

    sim = cos_sim(X_sorted)

    fig_hm = go.Figure(go.Heatmap(
        z=sim,
        x=labels_sorted,
        y=labels_sorted,
        colorscale='RdBu',
        zmid=0,
        hovertemplate='%{y}<br>%{x}<br>sim=%{z:.3f}<extra></extra>',
    ))
    fig_hm.update_layout(
        title=f'{nombre} — Matriz de similitud coseno (ordenada por línea)',
        height=700,
        xaxis=dict(showticklabels=False),
        yaxis=dict(showticklabels=False),
    )
    display(HTML(fig_hm.to_html(include_plotlyjs='cdn', full_html=False)))

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML

def plot_line_pca_plotly(emb, G, titulo):
    cols = [c for c in emb.columns if c.startswith('fo_') or c.startswith('so_')]
    X = emb[cols].values.astype(float)
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X)

    lines = [get_line(s, G) for s in emb.index]
    stations = list(emb.index)
    unique_lines = sorted(set(lines))

    fig = go.Figure()
    for line in unique_lines:
        mask = [l == line for l in lines]
        fig.add_trace(go.Scatter(
            x=coords[mask, 0],
            y=coords[mask, 1],
            mode='markers',
            name=line,
            text=[s for s, m in zip(stations, mask) if m],
            hovertemplate='<b>%{text}</b><br>PC1: %{x:.3f}<br>PC2: %{y:.3f}<extra></extra>',
            marker=dict(size=8, opacity=0.85),
        ))

    fig.update_layout(
        title=f'{titulo}<br><sup>PC1={pca.explained_variance_ratio_[0]*100:.1f}%  PC2={pca.explained_variance_ratio_[1]*100:.1f}%</sup>',
        xaxis_title='PC1',
        yaxis_title='PC2',
        legend_title='Línea',
        height=600,
        hovermode='closest',
    )
    return HTML(fig.to_html(include_plotlyjs='cdn', full_html=False))

plot_line_pca_plotly(emb_metro, G_metro, 'Metro — PCA embedding completo (FO + SO)')

In [ ]:
plot_line_pca_plotly(emb_cercanias, G_cercanias, 'Cercanías — PCA embedding completo (FO + SO)')

---
## 6 — Expansión a punto_acceso

In [ ]:
# Expandir de nivel estación a nivel punto_acceso
# (para combinar con LDA que trabaja a ese nivel)
ap_metro = model_metro.to_punto_acceso(G_metro)
ap_cercanias = model_cercanias.to_punto_acceso(G_cercanias)

print(f'Metro     — estaciones: {len(emb_metro):>3}  →  puntos acceso: {len(ap_metro):>3}')
print(f'Cercanías — estaciones: {len(emb_cercanias):>3}  →  puntos acceso: {len(ap_cercanias):>3}')
print()
print('Ejemplo metro:')
print(ap_metro.head(3).round(4))
print()
print('Verificar que el índice tiene formato punto_acceso:')
print(ap_metro.index[:5].tolist())

In [ ]:
# Verificar que el prefijo del índice es correcto
ok_metro = ap_metro.index.str.startswith('02_') | ap_metro.index.str.startswith('A9_')
ok_cerc  = ap_cercanias.index.str.startswith('04_')

print(f'Metro:     {ok_metro.sum()}/{len(ok_metro)} puntos acceso con prefijo 02_/A9_ → {"✓" if ok_metro.all() else "✗"}')
print(f'Cercanías: {ok_cerc.sum()}/{len(ok_cerc)} puntos acceso con prefijo 04_       → {"✓" if ok_cerc.all() else "✗"}')

In [ ]:
# CHECK: validación completa de la expansión a punto_acceso

n_emb_cols = 2 * model_metro.dim  # 64

for nombre, ap, G, model in [
    ('Metro',     ap_metro,     G_metro,     model_metro),
    ('Cercanías', ap_cercanias, G_cercanias, model_cercanias),
]:
    emb_cols = [c for c in model.embeddings_.columns
                if c.startswith('fo_') or c.startswith('so_')]
    ap_clean = ap[emb_cols].astype(float)

    # 1. Sin columnas extra (ej. _line colada por bug de referencia)
    extra_cols = [c for c in ap.columns if c not in emb_cols]
    ok_cols = len(extra_cols) == 0

    # 2. Sin NaN ni Inf
    ok_nan = not ap_clean.isnull().any().any() and np.isfinite(ap_clean.values).all()

    # 3. Sin vectores cero
    zero_mask = (ap_clean.values == 0).all(axis=1)
    ok_zero = not zero_mask.any()
    n_zero = zero_mask.sum()

    # 4. Cada punto_acceso hereda el vector correcto de su estación
    mismatches = 0
    for node, data in G.nodes(data=True):
        if node not in model.embeddings_.index:
            continue
        aps = data.get('access_point_codes', set())
        if isinstance(aps, str):
            aps = {aps}
        vec_station = model.embeddings_.loc[node, emb_cols].values.astype(float)
        for ap_code in aps:
            if ap_code not in ap_clean.index:
                mismatches += 1
                continue
            if not np.allclose(ap_clean.loc[ap_code].values, vec_station):
                mismatches += 1
    ok_map = mismatches == 0

    print(f'=== {nombre} ===')
    print(f'  Columnas correctas ({n_emb_cols}):    {"✓" if ok_cols else f"✗  extra={extra_cols}"}')
    print(f'  Sin NaN/Inf:                   {"✓" if ok_nan else "✗"}')
    print(f'  Sin vectores cero:             {"✓" if ok_zero else f"✗  ({n_zero} ceros)"}')
    print(f'  Herencia estación→AP correcta: {"✓" if ok_map else f"✗  ({mismatches} desajustes)"}')
    print()

---
## 7 — Guardar embeddings

In [ ]:
from pathlib import Path

out_dir = Path('output')
out_dir.mkdir(exist_ok=True)

# Nivel estación
emb_metro.to_parquet(out_dir / 'line_metro.parquet')
emb_metro.round(5).to_csv(out_dir / 'line_metro.csv')

emb_cercanias.to_parquet(out_dir / 'line_cercanias.parquet')
emb_cercanias.round(5).to_csv(out_dir / 'line_cercanias.csv')

# Nivel punto_acceso (para combinar con LDA)
ap_metro.to_parquet(out_dir / 'line_metro_ap.parquet')
ap_cercanias.to_parquet(out_dir / 'line_cercanias_ap.parquet')

print(f'Metro      → line_metro.parquet       {emb_metro.shape}')
print(f'Cercanías  → line_cercanias.parquet   {emb_cercanias.shape}')
print(f'Metro AP   → line_metro_ap.parquet    {ap_metro.shape}')
print(f'Cercanías AP → line_cercanias_ap.parquet {ap_cercanias.shape}')
print(f'Guardado en: {out_dir.resolve()}')